# 02 | Cross-Sectional Factor Screening

**Objective.** Rank the securities in the verified EUR-adjusted universe with a transparent combination of momentum, downside deviation, and historical volatility.

**Provenance.** Adapted from the uploaded operational Colab notebook (the 63/126-day factor screen). The **55% / 27% / 18%** weights are user-designed research parameters, **not Euronext's official scoring weights**. The score is a relative rank and **not** a forecast probability or expected return.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Run from the repository root, or from within notebooks/.
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
OUTPUT = ROOT / "reports" / "research_outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Repository root:", ROOT.resolve())
print("Data directory:", DATA.resolve())

In [ ]:
PRICE_FILE = DATA / "prices_eur_2026-11-04.csv"
if not PRICE_FILE.is_file():
    raise FileNotFoundError(
        f"Missing {PRICE_FILE}. Copy your previously validated EUR-adjusted "
        "price panel into data/ (or update PRICE_FILE). Do not commit private "
        "or third-party price datasets without reviewing distribution rights."
    )
prices = pd.read_csv(PRICE_FILE, index_col=0, parse_dates=True)
prices.index = pd.to_datetime(prices.index).normalize()
prices = prices.apply(pd.to_numeric, errors="coerce").sort_index()
if prices.index.has_duplicates or prices.columns.has_duplicates:
    raise ValueError("Duplicate timestamps or ticker identifiers")
if prices.isna().any().any() or (prices <= 0).any().any():
    raise ValueError("Missing or nonpositive values in the EUR price panel")
if prices.shape[0] < 254:
    raise ValueError("At least 254 prices required for a 252-return risk window")
returns = prices.pct_change(fill_method=None).dropna()
print(f"{prices.shape[1]} securities, {prices.shape[0]} dates; last = {prices.index[-1].date()}")

## 1. Statistical definition of factors

For adjusted price \(P_{i,t}\):

- \(M_{i,k}=P_{i,t}/P_{i,t-k}-1\), \(k\in\{63,126\}\).
- \(DD_i=\sqrt{252\,T^{-1}\sum_t\min(r_{i,t},0)^2}\).
- \(\sigma_i=\sqrt{252}\,\mathrm{sd}(r_{i,t})\), with sample standard deviation.

We first combine **percentile ranks** of the two momentum horizons in a 40/60 ratio, then use a 55/27/18 composite across momentum, inverse downside, and inverse volatility. One early exploratory cell in the raw notebook combined raw returns instead; this notebook follows the later operational percentile version (STEP 36A).

In [ ]:
lookback = returns.tail(252)
m63 = prices.iloc[-1] / prices.iloc[-64] - 1
m126 = prices.iloc[-1] / prices.iloc[-127] - 1
vol = lookback.std(ddof=1) * np.sqrt(252)
downside = np.sqrt(lookback.clip(upper=0).pow(2).mean() * 252)
rank_momentum = 0.40 * m63.rank(pct=True) + 0.60 * m126.rank(pct=True)
rank_downside = (-downside).rank(pct=True)
rank_volatility = (-vol).rank(pct=True)
score = 100 * (0.55 * rank_momentum + 0.27 * rank_downside + 0.18 * rank_volatility)
ranking = pd.DataFrame({
    "Score_100": score,
    "Momentum_63_pct": 100*m63,
    "Momentum_126_pct": 100*m126,
    "Downside_252_pct": 100*downside,
    "Vol_252_pct": 100*vol,
}).sort_values("Score_100", ascending=False)
ranking.index.name = "Ticker"
assert ranking["Score_100"].between(0,100).all()
print(ranking.head(20).round(3).to_string())

## 2. Correlation checks and candidate screening

A high-scoring individual stock can still worsen portfolio concentration or sector exposure. This diagnostic highlights highly correlated pairs among the best-ranked 20 names; sector exposure is handled in Notebook 03.

In [ ]:
top = ranking.head(min(20,len(ranking))).index.tolist()
corr = returns.tail(126)[top].corr()
pairs = []
for i, a in enumerate(top):
    for b in top[i+1:]:
        pairs.append((a, b, corr.loc[a,b]))
pairs = pd.DataFrame(pairs, columns=["Ticker_A","Ticker_B","Correlation_126"])
print(pairs.sort_values("Correlation_126", ascending=False).head(12).round(3).to_string(index=False))

## 3. Save a dated, auditable factor snapshot

This is an **in-sample** cross-sectional screen. It is not a historical portfolio return series. To validate a factor strategy, use point-in-time universe membership, walk-forward rebalancing, and transaction-cost assumptions.

In [ ]:
asof = prices.index[-1].strftime("%Y-%m-%d")
ranking.to_csv(OUTPUT / f"02_factor_ranking_{asof}.csv")
print("Factor ranking saved. As-of date:", asof)

## Discussion

- Why might a positive momentum signal reverse after earnings?
- How does a percentile score change when the investment universe changes?
- What biases arise when current survivors define the historical universe?
- Why should rankings calculated at time \(t\) never be used to simulate a trade before \(t\)?